# PM2.5 Episode-Aware Forecasting: Master Blend Pipeline

**Competition:** ANRF AI-SE Hack Phase 2 - Theme 2: Pollution Forecasting  
**Team:** Code4CleanAir  
**Score Achieved:** 0.8825  
**License:** ANRF Open License

---

## Overview

This notebook implements our winning approach for PM2.5 concentration forecasting with special emphasis on **episodic (extreme pollution) events**. The pipeline consists of:

1. **ResGRU-UNet Architecture**: Encoder-decoder with ConvGRU for temporal modeling
2. **Pure Quantile Refinement (PQR)**: Asymmetric loss targeting the 85th percentile
3. **Master Blend Inference**: Combining "Stable" and "Spike" expert models
4. **Curvature Correction**: Non-linear post-processing for extreme values

### Key Innovation: Asymmetric Quantile Loss

The core insight is that **under-predicting episodes is worse than over-predicting**. We use Pinball loss with q=0.85, which penalizes under-predictions 5.67x more than over-predictions:

```
Pinball(q=0.85): 
  - Under-prediction penalty: 0.85 × |error|
  - Over-prediction penalty:  0.15 × |error|
  - Asymmetry ratio: 5.67x
```

---

## Table of Contents

1. [Configuration & Setup](#1-configuration--setup)
2. [Data Pipeline](#2-data-pipeline)
3. [Model Architecture](#3-model-architecture)
4. [Loss Functions](#4-loss-functions)
5. [Training Loop](#5-training-loop)
6. [Master Blend Inference](#6-master-blend-inference)
7. [Main Execution](#7-main-execution)

---

## 1. Configuration & Setup

### 1.1 Imports and Environment

In [ ]:
import os
import gc
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

### 1.2 Base Configuration

These parameters define the model architecture and must match the checkpoint exactly.

In [ ]:
class Config:
    """
    Base configuration for data paths and model architecture.
    
    Architecture Parameters (must match checkpoint):
    - ENC_CHANNELS: Encoder channel progression [64, 128, 256]
    - GRU_HIDDEN: ConvGRU hidden dimensions (256)
    - DEC_CHANNELS: Decoder channel progression [128, 64, 32]
    """
    
    # Environment Detection
    IS_KAGGLE = os.path.exists('/kaggle/input')
    
    if IS_KAGGLE:
        DATA_ROOT   = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
        WORKING_DIR = '/kaggle/working'
    else:
        DATA_ROOT   = './aisehack-theme-2'
        WORKING_DIR = '.'

    RAW_DIR  = os.path.join(DATA_ROOT, 'raw')
    TEST_DIR = os.path.join(DATA_ROOT, 'test_in')

    # Training Data: 4 months from 2016
    MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']

    # Input Features (13 base + 3 engineered = 16 channels)
    FEATURES = [
        'cpm25',   # Chemical PM2.5 - PRIMARY TARGET VARIABLE
        'q2',      # 2m specific humidity
        't2',      # 2m temperature
        'u10',     # 10m U-wind component
        'v10',     # 10m V-wind component
        'pblh',    # Planetary boundary layer height
        'psfc',    # Surface pressure
        'swdown',  # Downward shortwave radiation
        'rain',    # Precipitation (log-transformed)
        'PM25',    # Additional PM2.5 source (zero-padded in dataset)
        'NOx',     # Nitrogen oxides (zero-padded in dataset)
        'SO2',     # Sulfur dioxide (zero-padded in dataset)
        'NH3',     # Ammonia (zero-padded in dataset)
    ]
    
    NUM_FEATURES        = len(FEATURES)           # 13
    NUM_INPUT_CHANNELS  = NUM_FEATURES + 3        # +wind_speed, hour_sin, hour_cos = 16
    
    # Feature Normalization Strategies
    MAX_SCALE_FEATURES     = {'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio'}
    LOG_TRANSFORM_FEATURES = {'rain'}

    # Temporal Configuration
    INPUT_HOURS  = 10   # Hours of input context
    OUTPUT_HOURS = 16   # Hours to forecast
    WINDOW_SIZE  = INPUT_HOURS + OUTPUT_HOURS  # 26 total

    # Spatial Dimensions (WRF-Chem grid)
    H = 140  # Height (latitude)
    W = 124  # Width (longitude)

    # Model Architecture (MUST MATCH CHECKPOINT)
    ENC_CHANNELS = [64, 128, 256]   # Encoder progression
    GRU_HIDDEN   = 256              # ConvGRU hidden state
    DEC_CHANNELS = [128, 64, 32]    # Decoder progression
    DROP_RATE    = 0.1              # Dropout rate

    # Episode Detection Thresholds
    EPISODE_STD_THRESHOLD = 2.0    # Std devs above mean
    EPISODE_MIN_CPM       = 1.0    # Minimum concentration

    # Inference Settings
    INFERENCE_BATCH_SIZE = 4
    NUM_TEST_SAMPLES     = 218

    SEED = 42

### 1.3 Pure Quantile Refinement (PQR) Configuration

Phase 4 training parameters for the "Spike Expert" model.

In [ ]:
class PQRConfig:
    """
    Phase 4: Pure Quantile Refinement Configuration
    
    Key Design Decisions:
    - QUANTILE=0.85: Targets extreme episodes (top 15%)
    - LR=1e-5: Ultra-low learning rate for fine-tuning
    - No Huber loss: 100% asymmetric pinball loss
    - BLEND: 70% spike + 30% stable for balanced predictions
    """
    
    # Training Parameters
    STRIDE       = 1      # Sliding window stride
    BATCH_SIZE   = 4      # Limited by GPU memory
    VAL_SPLIT    = 0.1    # 10% validation
    EPOCHS       = 5      # Maximum epochs
    LR           = 1e-5   # Ultra-low for fine-tuning
    WEIGHT_DECAY = 1e-4   # L2 regularization
    PATIENCE     = 3      # Early stopping patience

    # Pinball Loss Configuration
    QUANTILE = 0.85  # Target 85th percentile
    # Asymmetry: under-prediction cost = 0.85, over-prediction cost = 0.15
    # Ratio: 0.85 / 0.15 = 5.67x penalty for under-prediction

    # Master Blend Weights
    STABLE_WEIGHT = 0.30   # Phase 2 "Global" expert
    SPIKE_WEIGHT  = 0.70   # Phase 4 "Spike" expert

    # Curvature Correction Threshold
    CURVE_THRESHOLD = 100.0  # Apply correction above this value

    EPS = 1e-8  # Numerical stability

    # Checkpoint Paths
    WORKING_DIR      = Config.WORKING_DIR
    PQR_CKPT         = os.path.join(Config.WORKING_DIR, 'best_pqr_finetune.pt')
    STABLE_CKPT_PATH = '/kaggle/input/notebooks/kaori02/res-1fine/best_episodic_finetune.pt'
    PHASE3_CKPT_PATH = '/kaggle/input/notebooks/kaori02/res1-fine-quantile/best_quantile_finetune.pt'
    NORM_STATS_PATH  = '/kaggle/input/notebooks/kaori02/res-1/norm_stats_resgru.npz'

    @staticmethod
    def find_file(filename: str) -> str:
        """Resolve checkpoint file paths."""
        hardcoded = {
            'best_episodic_finetune.pt':  PQRConfig.STABLE_CKPT_PATH,
            'best_quantile_finetune.pt':  PQRConfig.PHASE3_CKPT_PATH,
            'norm_stats_resgru.npz':      PQRConfig.NORM_STATS_PATH,
        }
        return hardcoded.get(filename, os.path.join(Config.WORKING_DIR, filename))

### 1.4 Reproducibility

In [ ]:
def seed_everything(seed: int = Config.SEED):
    """Set random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True  # Enable for fixed input sizes

---

## 2. Data Pipeline

### 2.1 Normalization Statistics

We compute **per-pixel** mean and standard deviation for spatial normalization. This preserves local pollution patterns (e.g., industrial zones vs. rural areas).

In [ ]:
class NormStats:
    """
    Spatial normalization statistics for each feature.
    
    Normalization Strategies:
    - MAX_SCALE_FEATURES: Divide by global max (emissions data)
    - LOG_TRANSFORM_FEATURES: log1p then Z-score (rain)
    - Default: Per-pixel Z-score (Welford's algorithm)
    """
    
    def __init__(self):
        self.mean = {}  # {feature: (H, W) array}
        self.std = {}   # {feature: (H, W) array}

    def compute_from_raw(self, raw_dir, months, features):
        """Compute normalization statistics from training data."""
        print("Computing normalization statistics...")
        
        for feat in features:
            if feat in Config.MAX_SCALE_FEATURES:
                # Max-scaling for emissions (often all zeros)
                global_max = 0.0
                for month in months:
                    data = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    global_max = max(global_max, float(np.max(np.abs(data))))
                self.mean[feat] = np.zeros((Config.H, Config.W), dtype=np.float32)
                self.std[feat] = np.full((Config.H, Config.W), max(global_max, 1e-15), dtype=np.float32)

            elif feat in Config.LOG_TRANSFORM_FEATURES:
                # Log-transform then Z-score (for rain)
                s, sq, n = 0.0, 0.0, 0
                for month in months:
                    data = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    for t in range(data.shape[0]):
                        x = np.log1p(data[t].astype(np.float64))
                        s += x.sum()
                        sq += (x**2).sum()
                        n += x.size
                mean_ = s / n
                std_ = max(np.sqrt(sq / n - mean_**2), 1e-6)
                self.mean[feat] = np.full((Config.H, Config.W), mean_, dtype=np.float32)
                self.std[feat] = np.full((Config.H, Config.W), std_, dtype=np.float32)

            else:
                # Per-pixel Z-score using Welford's online algorithm
                cnt = 0
                m_acc = np.zeros((Config.H, Config.W), np.float64)
                m2 = np.zeros((Config.H, Config.W), np.float64)
                
                for month in months:
                    data = np.load(os.path.join(raw_dir, month, f'{feat}.npy'), mmap_mode='r')
                    for t in range(data.shape[0]):
                        x = data[t].astype(np.float64)
                        cnt += 1
                        d = x - m_acc
                        m_acc += d / cnt
                        m2 += d * (x - m_acc)
                        
                self.mean[feat] = m_acc.astype(np.float32)
                self.std[feat] = np.maximum(
                    np.sqrt(m2 / max(cnt-1, 1)).astype(np.float32), 
                    1e-6
                )
            print(f"  {feat}: done")

    def normalize(self, data, feat_name):
        """Apply normalization to input data."""
        if feat_name in Config.LOG_TRANSFORM_FEATURES:
            data = np.log1p(data)
        return (data - self.mean[feat_name]) / self.std[feat_name]

    def save(self, path):
        """Save statistics to .npz file."""
        np.savez(path, 
                 features=list(self.mean.keys()),
                 **{f'mean_{k}': v for k, v in self.mean.items()},
                 **{f'std_{k}': v for k, v in self.std.items()})

    def load(self, path):
        """Load statistics from .npz file."""
        d = np.load(path)
        for feat in list(d['features']):
            self.mean[feat] = d[f'mean_{feat}']
            self.std[feat] = d[f'std_{feat}']

### 2.2 Engineered Features

We add three engineered features:
- **Wind Speed**: Dispersion indicator (sqrt(u10^2 + v10^2))
- **Hour Sin/Cos**: Cyclical encoding of time-of-day

In [ ]:
def compute_engineered_features(u10, v10, hour_indices):
    """
    Compute engineered features from wind components and hour.
    
    Args:
        u10: U-wind component (T, H, W)
        v10: V-wind component (T, H, W)
        hour_indices: Hour of day for each timestep (T,)
    
    Returns:
        wind_speed: Magnitude of wind vector (T, H, W)
        hour_sin: Cyclical hour encoding (T, H, W)
        hour_cos: Cyclical hour encoding (T, H, W)
    """
    # Wind speed as dispersion indicator
    wind_speed = np.sqrt(u10**2 + v10**2).astype(np.float32)
    
    T, H, W = u10.shape
    hours = hour_indices.astype(np.float32)
    
    # Cyclical encoding: sin/cos preserve continuity (23:00 -> 00:00)
    hour_sin = np.sin(2 * np.pi * hours / 24.0)[:, None, None] * np.ones((1, H, W), np.float32)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)[:, None, None] * np.ones((1, H, W), np.float32)
    
    return wind_speed, hour_sin, hour_cos

### 2.3 Dataset Classes

In [ ]:
class PM25TrainDataset(Dataset):
    """
    Training dataset with sliding window sampling.
    
    Returns:
        inputs: (C, T, H, W) normalized input features
        target_log: (16, H, W) log1p target values
        target_raw: (16, H, W) raw target values (for metrics)
        last_log: (H, W) last input frame in log-space (for residual prediction)
        ctx_raw: (10, H, W) input context (for episode detection)
    """
    
    def __init__(self, raw_dir, months, features, norm_stats, stride=1):
        self.features = features
        self.norm_stats = norm_stats
        self.samples = []
        self.data_paths = {}

        for m_idx, month in enumerate(months):
            T = np.load(os.path.join(raw_dir, month, 'cpm25.npy'), mmap_mode='r').shape[0]
            n = (T - Config.WINDOW_SIZE) // stride + 1
            for i in range(n):
                self.samples.append((m_idx, i * stride))
            for feat in features:
                self.data_paths[(m_idx, feat)] = os.path.join(raw_dir, month, f'{feat}.npy')

        print(f"  TrainDataset: {len(self.samples)} samples, stride={stride}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        m_idx, start = self.samples[idx]
        input_list = []
        u10_norm, v10_norm = None, None

        # Load and normalize base features
        for feat in self.features:
            chunk = np.load(
                self.data_paths[(m_idx, feat)], 
                mmap_mode='r'
            )[start:start+Config.INPUT_HOURS].astype(np.float32)
            chunk = self.norm_stats.normalize(chunk, feat)
            input_list.append(chunk)
            
            if feat == 'u10':
                u10_norm = chunk
            elif feat == 'v10':
                v10_norm = chunk

        # Add engineered features
        ws, hs, hc = compute_engineered_features(
            u10_norm, v10_norm, 
            np.arange(start, start + Config.INPUT_HOURS) % 24
        )
        input_list.extend([ws, hs, hc])
        inputs = np.stack(input_list, axis=0)  # (C, T, H, W)

        # Load targets
        cpm_raw = np.load(self.data_paths[(m_idx, 'cpm25')], mmap_mode='r')
        target_raw = cpm_raw[start + Config.INPUT_HOURS:start + Config.WINDOW_SIZE].astype(np.float32)
        target_log = np.log1p(target_raw)
        last_log = np.log1p(cpm_raw[start + Config.INPUT_HOURS - 1].astype(np.float32))
        ctx_raw = cpm_raw[start:start + Config.INPUT_HOURS].astype(np.float32)

        return (
            torch.from_numpy(inputs),
            torch.from_numpy(target_log),
            torch.from_numpy(target_raw),
            torch.from_numpy(last_log),
            torch.from_numpy(ctx_raw)
        )


class PM25TestDataset(Dataset):
    """
    Test dataset (218 pre-sampled windows).
    
    Returns:
        inputs: (C, T, H, W) normalized input features
        last_log: (H, W) last input frame in log-space
    """
    
    def __init__(self, test_dir, features, norm_stats):
        self.features = features
        self.norm_stats = norm_stats
        self.data = {
            f: np.load(os.path.join(test_dir, f'{f}.npy'), mmap_mode='c') 
            for f in features
        }
        self.n = self.data[features[0]].shape[0]
        print(f"  TestDataset: {self.n} samples")

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        input_list = []
        u10_norm, v10_norm = None, None
        
        for feat in self.features:
            chunk = self.data[feat][idx].astype(np.float32)
            chunk = self.norm_stats.normalize(chunk, feat)
            input_list.append(chunk)
            
            if feat == 'u10':
                u10_norm = chunk
            elif feat == 'v10':
                v10_norm = chunk

        ws, hs, hc = compute_engineered_features(
            u10_norm, v10_norm,
            np.arange(Config.INPUT_HOURS).astype(np.float32)
        )
        input_list.extend([ws, hs, hc])
        inputs = np.stack(input_list, axis=0)
        last_log = np.log1p(self.data['cpm25'][idx].astype(np.float32)[-1])
        
        return torch.from_numpy(inputs), torch.from_numpy(last_log)

---

## 3. Model Architecture

### 3.1 Building Blocks

In [ ]:
class ConvBlock(nn.Module):
    """
    Residual convolutional block with GroupNorm and GELU activation.
    
    Architecture:
        x -> Conv3x3 -> GN -> GELU -> Dropout -> Conv3x3 -> GN -> GELU -> + -> out
        |                                                                ^
        +------------------------ Skip (1x1 if channels differ) ---------+
    """
    
    def __init__(self, in_ch, out_ch, drop_rate=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(min(32, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(min(32, out_ch), out_ch)
        self.drop = nn.Dropout2d(drop_rate) if drop_rate > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        r = self.skip(x)
        x = F.gelu(self.gn1(self.conv1(x)))
        x = self.drop(x)
        x = F.gelu(self.gn2(self.conv2(x)))
        return x + r


class ConvGRUCell(nn.Module):
    """
    Convolutional GRU cell for spatiotemporal modeling.
    
    Unlike standard GRU, this operates on 2D feature maps,
    preserving spatial structure while modeling temporal dynamics.
    
    Equations:
        r = sigmoid(W_xr * x + W_hr * h + b_r)  # Reset gate
        u = sigmoid(W_xu * x + W_hu * h + b_u)  # Update gate
        c = tanh(W_xc * x + W_hc * (r * h) + b_c)  # Candidate
        h' = (1 - u) * h + u * c  # New hidden state
    """
    
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.conv_gates = nn.Conv2d(
            input_dim + hidden_dim, 2 * hidden_dim, 
            kernel_size, padding=p, bias=True
        )
        self.conv_candidate = nn.Conv2d(
            input_dim + hidden_dim, hidden_dim, 
            kernel_size, padding=p, bias=True
        )
        self.gn_gates = nn.GroupNorm(min(32, 2 * hidden_dim), 2 * hidden_dim)
        self.gn_cand = nn.GroupNorm(min(32, hidden_dim), hidden_dim)

    def forward(self, x, h):
        comb = torch.cat([x, h], dim=1)
        gates = torch.sigmoid(self.gn_gates(self.conv_gates(comb)))
        r, u = gates.chunk(2, dim=1)
        cand = torch.tanh(self.gn_cand(self.conv_candidate(torch.cat([x, r * h], dim=1))))
        return (1 - u) * h + u * cand

### 3.2 ResGRU-UNet Model

In [ ]:
class ResGRUUNet(nn.Module):
    """
    Residual GRU U-Net for PM2.5 forecasting.
    
    Architecture:
        1. Encoder: 3-level conv blocks with pooling
        2. Bottleneck: ConvGRU for temporal dynamics
        3. Decoder: U-Net style with skip connections
        4. Autoregressive: Runs 16 times for 16-hour forecast
    
    Key Design:
        - Predicts RESIDUALS in log-space: pred = last_frame_log + delta
        - Skip connections preserve spatial details for local hotspots
        - ConvGRU hidden state accumulates information across output steps
    """
    
    def __init__(self, cfg=Config):
        super().__init__()
        self.cfg = cfg
        C_in = cfg.NUM_INPUT_CHANNELS * cfg.INPUT_HOURS  # 16 * 10 = 160
        enc_ch = cfg.ENC_CHANNELS  # [64, 128, 256]
        dec_ch = cfg.DEC_CHANNELS  # [128, 64, 32]
        drop = cfg.DROP_RATE

        # Encoder
        self.enc0 = ConvBlock(C_in, enc_ch[0], drop)       # 160 -> 64
        self.pool0 = nn.MaxPool2d(2)                        # 140x124 -> 70x62
        self.enc1 = ConvBlock(enc_ch[0], enc_ch[1], drop)  # 64 -> 128
        self.pool1 = nn.MaxPool2d(2)                        # 70x62 -> 35x31
        self.enc2 = ConvBlock(enc_ch[1], enc_ch[2], drop)  # 128 -> 256
        self.pool2 = nn.AvgPool2d(2, ceil_mode=True)        # 35x31 -> 18x16

        # Bottleneck with ConvGRU
        self.bottleneck = ConvBlock(enc_ch[2], cfg.GRU_HIDDEN, drop)  # 256 -> 256
        self.convgru = ConvGRUCell(cfg.GRU_HIDDEN, cfg.GRU_HIDDEN, kernel_size=3)
        self.gru_proj = nn.Conv2d(cfg.GRU_HIDDEN, enc_ch[2], 1, bias=False)  # 256 -> 256

        # Decoder with skip connections
        self.up2 = nn.ConvTranspose2d(enc_ch[2], dec_ch[0], 2, stride=2)
        self.dec2 = ConvBlock(dec_ch[0] + enc_ch[2], dec_ch[0], drop)  # 128+256 -> 128
        self.up1 = nn.ConvTranspose2d(dec_ch[0], dec_ch[1], 2, stride=2)
        self.dec1 = ConvBlock(dec_ch[1] + enc_ch[1], dec_ch[1], drop)  # 64+128 -> 64
        self.up0 = nn.ConvTranspose2d(dec_ch[1], dec_ch[2], 2, stride=2)
        self.dec0 = ConvBlock(dec_ch[2] + enc_ch[0], dec_ch[2], drop)  # 32+64 -> 32

        # Output head
        self.head = nn.Sequential(
            nn.Conv2d(dec_ch[2], dec_ch[2], 3, padding=1, bias=False),
            nn.GroupNorm(min(32, dec_ch[2]), dec_ch[2]),
            nn.GELU(),
            nn.Conv2d(dec_ch[2], 1, 1),
        )
        self._init_weights()

    def _init_weights(self):
        """Initialize weights: Kaiming for conv, zeros for final layer."""
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.GroupNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
        # Zero-init final layer for stable residual learning
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def _pad(self, x, target):
        """Pad/crop feature map to match target size for skip connections."""
        dh = target.shape[2] - x.shape[2]
        dw = target.shape[3] - x.shape[3]
        if dh > 0 or dw > 0:
            return F.pad(x, [0, dw, 0, dh])
        if dh < 0 or dw < 0:
            return x[:, :, :target.shape[2], :target.shape[3]]
        return x

    def forward(self, x, last_frame_log):
        """
        Forward pass with autoregressive decoding.
        
        Args:
            x: Input tensor (B, C, T, H, W)
            last_frame_log: Last input frame in log-space (B, H, W)
        
        Returns:
            Predictions in log-space (B, 16, H, W)
        """
        B, C, T, H, W = x.shape
        x = x.reshape(B, C * T, H, W)  # Flatten time into channels
        
        # Encoder
        e0 = self.enc0(x)
        e1 = self.enc1(self.pool0(e0))
        e2 = self.enc2(self.pool1(e1))
        
        # Bottleneck
        bn = self.bottleneck(self.pool2(e2))
        h = bn  # Initialize GRU hidden state
        
        # Autoregressive decoding: 16 steps
        outputs = []
        for _ in range(self.cfg.OUTPUT_HOURS):
            h = self.convgru(bn, h)  # Update hidden state
            d = self.gru_proj(h)     # Project to decoder input
            
            # Decoder with skip connections
            d2 = self.dec2(torch.cat([self._pad(self.up2(d), e2), e2], dim=1))
            d1 = self.dec1(torch.cat([self._pad(self.up1(d2), e1), e1], dim=1))
            d0 = self.dec0(torch.cat([self._pad(self.up0(d1), e0), e0], dim=1))
            
            outputs.append(self.head(d0).squeeze(1))
        
        # Stack outputs and add residual from last frame
        deltas = torch.stack(outputs, dim=1)  # (B, 16, H, W)
        return last_frame_log.unsqueeze(1) + deltas

---

## 4. Loss Functions

### 4.1 Pure Pinball Loss

The key innovation: 100% asymmetric loss targeting the 85th percentile.

In [ ]:
class PurePinballLoss(nn.Module):
    """
    100% Asymmetric Pinball Loss at q=0.85 (log-space).
    
    Under-prediction cost: q * |residual| = 0.85
    Over-prediction cost: (1-q) * |residual| = 0.15
    Asymmetry ratio: 0.85 / 0.15 = 5.67x
    
    This forces the model to over-predict rather than under-predict
    extreme values, directly optimizing for Episode SMAPE.
    """
    
    def __init__(self):
        super().__init__()
        self.q = PQRConfig.QUANTILE  # 0.85
        self.eps = PQRConfig.EPS

    def forward(self, pred_log, target_log, target_raw, last_frame_log, context_cpm_raw):
        """
        Compute pinball loss and monitoring metrics.
        
        Args:
            pred_log: Predictions in log-space (B, 16, H, W)
            target_log: Targets in log-space (B, 16, H, W)
            target_raw: Targets in raw space (for metrics)
            last_frame_log: Not used (for API compatibility)
            context_cpm_raw: Input context (for episode detection)
        
        Returns:
            loss: Pinball loss value
            metrics: Dict of monitoring metrics
        """
        # Pinball loss in log-space
        residual = target_log - pred_log  # Positive = under-prediction
        pinball_loss = torch.mean(torch.max(
            self.q * residual,        # Under-prediction penalty
            (self.q - 1) * residual   # Over-prediction penalty
        ))

        # Compute monitoring metrics (no gradient)
        with torch.no_grad():
            pred_raw = F.relu(torch.expm1(pred_log.clamp(max=15)))

            # Episode detection
            b_mean = context_cpm_raw.mean(dim=1, keepdim=True)
            b_std = context_cpm_raw.std(dim=1, keepdim=True) + self.eps
            is_ep = (
                (target_raw > b_mean + Config.EPISODE_STD_THRESHOLD * b_std)
                & (target_raw > Config.EPISODE_MIN_CPM)
            )

            # SMAPE calculation
            numer = torch.abs(pred_raw - target_raw)
            denom = (torch.abs(pred_raw) + torch.abs(target_raw)) * 0.5 + self.eps
            smape = (numer / denom).mean().item()

            # Episode SMAPE
            ep_f = is_ep.float()
            n_ep = ep_f.sum().item()
            ep_smape = ((numer / denom) * ep_f).sum().item() / max(n_ep, 1.0)

            # Episode correlation
            ep_corr = 0.0
            if n_ep >= 10:
                flat = is_ep.reshape(-1)
                p_e = pred_raw.reshape(-1)[flat]
                y_e = target_raw.reshape(-1)[flat]
                pc = p_e - p_e.mean()
                yc = y_e - y_e.mean()
                cov = (pc * yc).sum()
                ep_corr = (cov / (torch.sqrt((pc**2).sum() + self.eps) *
                                  torch.sqrt((yc**2).sum() + self.eps))).item()

        return pinball_loss, {
            'global_smape': smape,
            'episode_smape': ep_smape,
            'episode_corr': ep_corr,
            'n_episodes': n_ep,
        }

---

## 5. Training Loop

In [ ]:
@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validation loop with metrics computation."""
    model.eval()
    total, metrics_acc, n = 0.0, {}, 0
    
    for inputs, t_log, t_raw, lf_log, ctx in loader:
        inputs = inputs.to(device, non_blocking=True)
        t_log = t_log.to(device, non_blocking=True)
        t_raw = t_raw.to(device, non_blocking=True)
        lf_log = lf_log.to(device, non_blocking=True)
        ctx = ctx.to(device, non_blocking=True)
        
        with autocast(dtype=torch.float16):
            pred_log = model(inputs, lf_log)
            loss, metrics = criterion(pred_log.float(), t_log, t_raw, lf_log, ctx)
        
        total += loss.item()
        for k, v in metrics.items():
            metrics_acc[k] = metrics_acc.get(k, 0.0) + v
        n += 1
    
    avg = total / max(n, 1)
    for k in metrics_acc:
        metrics_acc[k] /= max(n, 1)
    return avg, metrics_acc


def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    """Training loop for one epoch with gradient scaling."""
    model.train()
    total, metrics_acc, n = 0.0, {}, 0
    
    for i, (inputs, t_log, t_raw, lf_log, ctx) in enumerate(loader):
        inputs = inputs.to(device, non_blocking=True)
        t_log = t_log.to(device, non_blocking=True)
        t_raw = t_raw.to(device, non_blocking=True)
        lf_log = lf_log.to(device, non_blocking=True)
        ctx = ctx.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with autocast(dtype=torch.float16):
            pred_log = model(inputs, lf_log)
            loss, metrics = criterion(pred_log.float(), t_log, t_raw, lf_log, ctx)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total += loss.item()
        for k, v in metrics.items():
            metrics_acc[k] = metrics_acc.get(k, 0.0) + v
        n += 1
        
        if i % 50 == 0:
            print(f"  Ep{epoch} [{i}/{len(loader)}] "
                  f"pinball={loss.item():.4f}  "
                  f"ep_smape={metrics.get('episode_smape', 0):.4f}  "
                  f"ep_corr={metrics.get('episode_corr', 0):.4f}")
    
    avg = total / max(n, 1)
    for k in metrics_acc:
        metrics_acc[k] /= max(n, 1)
    return avg, metrics_acc

---

## 6. Master Blend Inference

### 6.1 Single Model Inference

In [ ]:
@torch.no_grad()
def run_inference_single(model, norm_stats, device) -> np.ndarray:
    """Run inference with a single model. Returns (N, H, W, T) array."""
    model.eval()
    cfg = Config
    preds = np.zeros(
        (cfg.NUM_TEST_SAMPLES, cfg.H, cfg.W, cfg.OUTPUT_HOURS), 
        dtype=np.float32
    )

    test_ds = PM25TestDataset(cfg.TEST_DIR, cfg.FEATURES, norm_stats)
    test_loader = DataLoader(
        test_ds, 
        batch_size=cfg.INFERENCE_BATCH_SIZE,
        shuffle=False, 
        num_workers=0, 
        pin_memory=True
    )
    
    idx = 0
    for batch_x, batch_lf in test_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_lf = batch_lf.to(device, non_blocking=True)
        bs = batch_x.shape[0]

        with autocast(dtype=torch.float16):
            pred_log = model(batch_x, batch_lf).float()

        pred_raw = F.relu(torch.expm1(pred_log.clamp(max=15)))
        preds[idx:idx+bs] = pred_raw.permute(0, 2, 3, 1).cpu().numpy()
        idx += bs

        if idx % 50 == 0 or idx >= cfg.NUM_TEST_SAMPLES:
            print(f"    {idx}/{cfg.NUM_TEST_SAMPLES}")

        del batch_x, batch_lf, pred_log, pred_raw
        torch.cuda.empty_cache()

    return preds

### 6.2 Master Blend with Curvature Correction

In [ ]:
def run_master_blend_inference(norm_stats, device, pqr_ckpt_path: str) -> np.ndarray:
    """
    OOM-safe dual-model inference with blending and post-processing.
    
    Pipeline:
        1. Load Stable model (0.8768) -> infer -> store -> delete from GPU
        2. Load PQR model -> infer -> store -> delete from GPU
        3. Blend: 0.30 x stable + 0.70 x spike
        4. Apply Curvature Correction on peaks > 100
        5. Return final array (N, H, W, T)
    
    Curvature Correction Formula:
        For preds > 100: multiply by (1 + (preds/1000)^2)
        - At 100 ug/m3 -> x1.01 (+1%)
        - At 300 ug/m3 -> x1.09 (+9%)
        - At 600 ug/m3 -> x1.36 (+36%)
        - At 1000 ug/m3 -> x2.00 (+100%)
    """
    cfg = Config
    pqr = PQRConfig

    # ---- Pass 1: Stable expert (0.8768) ----
    print("\n  [Blend] Pass 1/2 - Stable expert (0.8768) ...")
    stable_path = pqr.find_file('best_episodic_finetune.pt')
    assert os.path.exists(stable_path), f"Stable checkpoint not found: {stable_path}"

    model_stable = ResGRUUNet(cfg).to(device)
    ckpt_stable = torch.load(stable_path, map_location=device, weights_only=True)
    model_stable.load_state_dict(ckpt_stable['model_state_dict'])
    print(f"    Loaded stable checkpoint: {stable_path}")

    preds_stable = run_inference_single(model_stable, norm_stats, device)

    del model_stable, ckpt_stable
    gc.collect()
    torch.cuda.empty_cache()
    print(f"    Stable range: [{preds_stable.min():.1f}, {preds_stable.max():.1f}]")

    # ---- Pass 2: PQR Spike expert ----
    print("\n  [Blend] Pass 2/2 - PQR Spike expert ...")
    assert os.path.exists(pqr_ckpt_path), f"PQR checkpoint not found: {pqr_ckpt_path}"

    model_spike = ResGRUUNet(cfg).to(device)
    ckpt_spike = torch.load(pqr_ckpt_path, map_location=device, weights_only=True)
    model_spike.load_state_dict(ckpt_spike['model_state_dict'])
    print(f"    Loaded PQR checkpoint: {pqr_ckpt_path}")

    preds_spike = run_inference_single(model_spike, norm_stats, device)

    del model_spike, ckpt_spike
    gc.collect()
    torch.cuda.empty_cache()
    print(f"    Spike range: [{preds_spike.min():.1f}, {preds_spike.max():.1f}]")

    # ---- Blend ----
    print(f"\n  [Blend] Combining: {pqr.STABLE_WEIGHT} x stable + {pqr.SPIKE_WEIGHT} x spike ...")
    preds = pqr.STABLE_WEIGHT * preds_stable + pqr.SPIKE_WEIGHT * preds_spike
    del preds_stable, preds_spike
    gc.collect()
    print(f"    Blended range: [{preds.min():.1f}, {preds.max():.1f}]")

    # ---- Curvature Correction ----
    print("\n  [Blend] Applying Curvature Correction (peaks > 100) ...")
    stretch = 1.0 + (preds / 1000.0) ** 2
    preds = np.where(preds > pqr.CURVE_THRESHOLD, preds * stretch, preds)
    n_stretched = (preds > pqr.CURVE_THRESHOLD).sum()
    print(f"    Stretched {n_stretched:,} pixels ({100 * n_stretched / preds.size:.2f}%)")
    print(f"    Final range: [{preds.min():.1f}, {preds.max():.1f}]  mean={preds.mean():.3f}")

    return preds

---

## 7. Main Execution

In [ ]:
def main():
    """
    Main execution pipeline:
        1. Load normalization statistics
        2. Create datasets
        3. Load Phase 3 checkpoint
        4. PQR fine-tuning with early stopping
        5. Master Blend inference
    """
    seed_everything()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    cfg = Config
    pqr = PQRConfig

    # ---- Step 1: Norm stats ----
    print("\n" + "="*60)
    print("STEP 1: Loading normalization statistics")
    print("="*60)
    norm_path = pqr.find_file('norm_stats_resgru.npz')
    norm_stats = NormStats()
    if os.path.exists(norm_path):
        print(f"  Loading from {norm_path}")
        norm_stats.load(norm_path)
    else:
        print("  No cache found - recomputing...")
        norm_stats.compute_from_raw(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES)
        norm_stats.save(os.path.join(pqr.WORKING_DIR, 'norm_stats_resgru.npz'))
    gc.collect()

    # ---- Step 2: Datasets ----
    print("\n" + "="*60)
    print(f"STEP 2: Creating datasets (STRIDE={pqr.STRIDE})")
    print("="*60)
    full_ds = PM25TrainDataset(cfg.RAW_DIR, cfg.MONTHS, cfg.FEATURES, norm_stats, stride=pqr.STRIDE)
    n_val = int(len(full_ds) * pqr.VAL_SPLIT)
    n_train = len(full_ds) - n_val
    train_ds, val_ds = torch.utils.data.random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(cfg.SEED)
    )
    print(f"  Train: {n_train}  Val: {n_val}")

    train_loader = DataLoader(
        train_ds, batch_size=pqr.BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=pqr.BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True, persistent_workers=True
    )

    # ---- Step 3: Load Phase 3 checkpoint ----
    print("\n" + "="*60)
    print("STEP 3: Loading Phase 3 checkpoint (0.8803)")
    print("="*60)
    phase3_path = pqr.find_file('best_quantile_finetune.pt')
    assert os.path.exists(phase3_path), f"Phase 3 checkpoint not found: {phase3_path}"
    
    model = ResGRUUNet(cfg).to(device)
    ckpt = torch.load(phase3_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"  Loaded: {phase3_path}")
    print(f"  epoch={ckpt.get('epoch', '?')}  ep_smape={ckpt.get('episode_smape', '?')}")

    # ---- Step 4: PQR fine-tuning ----
    print("\n" + "="*60)
    print(f"STEP 4: Pure Quantile Refinement  q={pqr.QUANTILE}  LR={pqr.LR}  Epochs={pqr.EPOCHS}")
    print("="*60)
    print("  Loss = 100% Pinball(q=0.85) - Huber removed")

    criterion = PurePinballLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=pqr.LR, weight_decay=pqr.WEIGHT_DECAY)
    scaler = GradScaler()

    best_ep_smape = float('inf')
    best_epoch = 0
    patience_count = 0

    for epoch in range(1, pqr.EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_m = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch)
        vl_loss, vl_m = validate(model, val_loader, criterion, device)
        ep_smape = vl_m.get('episode_smape', float('inf'))
        ep_corr = vl_m.get('episode_corr', 0.0)

        print(f"\nEpoch {epoch}/{pqr.EPOCHS}  ({time.time()-t0:.0f}s)")
        print(f"  Train | pinball={tr_loss:.4f}  "
              f"global={tr_m.get('global_smape', 0):.4f}  "
              f"ep_smape={tr_m.get('episode_smape', 0):.4f}  "
              f"ep_corr={tr_m.get('episode_corr', 0):.4f}")
        print(f"  Val   | pinball={vl_loss:.4f}  "
              f"global={vl_m.get('global_smape', 0):.4f}  "
              f"ep_smape={ep_smape:.4f}  "
              f"ep_corr={ep_corr:.4f}  "
              f"n_ep={vl_m.get('n_episodes', 0):.0f}")

        if ep_smape < best_ep_smape:
            best_ep_smape = ep_smape
            best_epoch = epoch
            patience_count = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_loss': vl_loss,
                'episode_smape': ep_smape,
                'episode_corr': ep_corr,
                'val_metrics': vl_m,
            }, pqr.PQR_CKPT)
            print(f"  -> Saved best  episode_smape={ep_smape:.4f}  ep_corr={ep_corr:.4f}")
        else:
            patience_count += 1
            print(f"  -> No improvement (best={best_ep_smape:.4f} @ epoch {best_epoch})  "
                  f"patience {patience_count}/{pqr.PATIENCE}")
            if patience_count >= pqr.PATIENCE:
                print(f"\n  Early stopping triggered after {epoch} epochs.")
                break

    print(f"\nPQR fine-tuning complete.  Best Episode SMAPE: {best_ep_smape:.4f} @ epoch {best_epoch}")

    # ---- Step 5: Master Blend Inference ----
    print("\n" + "="*60)
    print("STEP 5: Master Blend Inference")
    print("="*60)
    print(f"  Stable weight: {pqr.STABLE_WEIGHT}  ({pqr.STABLE_CKPT_PATH})")
    print(f"  Spike weight:  {pqr.SPIKE_WEIGHT}  ({pqr.PQR_CKPT})")

    del model, train_loader, val_loader, train_ds, val_ds, full_ds
    gc.collect()
    torch.cuda.empty_cache()

    preds = run_master_blend_inference(norm_stats, device, pqr.PQR_CKPT)

    output_path = os.path.join(pqr.WORKING_DIR, 'preds.npy')
    np.save(output_path, preds)

    print("\n" + "="*60)
    print("PHASE 4 MASTER BLEND COMPLETE")
    print(f"  PQR Checkpoint: {pqr.PQR_CKPT}")
    print(f"  Predictions:    {output_path}  shape={preds.shape}")
    print(f"  Best Ep SMAPE:  {best_ep_smape:.4f} @ epoch {best_epoch}")
    print("="*60)


if __name__ == '__main__':
    main()

---

## License

This work is released under the **ANRF Open License** for AI-SE Hack 2026.

See [LICENSE](./LICENSE) for full terms.

---

**Team:** Code4CleanAir  
**Competition:** ANRF AI-SE Hack Phase 2 - Theme 2  
**Score:** 0.8825